# Make training batches in Colab

Makes the batch folders the fine-tuning notebook reads (`MyDrive/surya/batches/epoch_001`, `epoch_002`, ...),
entirely in Colab: our generator, SynthDoc, OCRSmith, scan-simulator and Chromium are set up here.

**Once, before the first run: let Colab read the private repository.**

1. On GitHub: *Settings -> Developer settings -> Personal access tokens -> Fine-grained tokens -> Generate new
   token*. Repository access: *Only select repositories* -> `OCRSynthGen`. Permissions: *Contents: Read-only*.
2. In Colab: the key icon on the left (*Secrets*) -> *Add new secret*, name `GITHUB_TOKEN`, value: the token,
   and switch on *Notebook access*.

**Then:** *Runtime -> Run all*. A CPU runtime is enough (no GPU needed). It can run at the same time as the
fine-tuning notebook, in another tab: there set `WAIT_MINUTES` to, for example, 120, so training waits for each
next folder.

The first run also builds the photo pool (about 3,200 photos) and the Wikipedia text pool and keeps them in
`MyDrive/surya/cache/`; later runs take them from there.

In [ ]:
from pathlib import Path

BRANCH = "surya"
EPOCHS = [1, 2]            # makes batches/epoch_001, epoch_002 (each epoch's seed is its number)
PAGES = 1000               # pages per epoch folder
TASK = "full_ocr+block_ocr"
TASHKEEL = "none"
WORKERS = 0                # pages made at the same time; 0: one per CPU core
MS_FONTS = False           # True: install Microsoft's core fonts (Arial, Georgia, Times New Roman, Verdana...) so
                           # English pages use more fonts; this accepts Microsoft's licence for those fonts
DRIVE = Path("/content/drive/MyDrive/surya")

import subprocess
from google.colab import drive
drive.mount("/content/drive")


def sh(command):
    """Runs a shell command, shows its output as it comes, and stops here if it fails (Colab's `!` does not)."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"This step failed (exit code {process.returncode}); the message is above:\n{command}")

## 1. The code

Our repository (branch `surya`, with the token from *Secrets*), and the three tools it uses, each at the
exact version it was tested with. They sit side by side in `/content/ocr/`, as the generator expects.

In [ ]:
import base64
from google.colab import userdata

WORK = Path("/content/ocr")
REPO = WORK / "arabic-template-bank"
WORK.mkdir(exist_ok=True)

token = userdata.get("GITHUB_TOKEN")
header = "Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
if REPO.exists():
    git = ["git", "-C", str(REPO), "-c", f"http.extraHeader={header}", "pull", "-q"]
else:
    git = ["git", "-c", f"http.extraHeader={header}", "clone", "-q", "--depth", "1", "-b", BRANCH,
           "https://github.com/Yousef10p/OCRSynthGen.git", str(REPO)]
result = subprocess.run(git, capture_output=True, text=True)
if result.returncode:  # the message only, never the command (it holds the token)
    raise RuntimeError("Could not get the repository (is GITHUB_TOKEN right?): " + result.stderr.replace(header, "***"))
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

TOOLS = {"synthdoc": ("https://github.com/MatterWaveAI/synthdoc.git", "1c4506dccf87984f3d8e7fdfe560fe125bd1995a"),
         "ocrsmith": ("https://github.com/atlasia-ma/OCRSmith.git", "7ebaef82a8883499f76bd56001eab01e61ae707c"),
         "scan-simulator": ("https://github.com/s1mb1o/scan-simulator.git", "5befedffa2c43cca599887d512785322b4b0f3c6")}
for name, (url, commit) in TOOLS.items():
    if not (WORK / name / ".git").exists():
        sh(f"git init -q {WORK / name} && cd {WORK / name} && git fetch -q --depth 1 {url} {commit} && git checkout -q FETCH_HEAD")
    print(name, subprocess.run(["git", "-C", str(WORK / name), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())

## 2. Python, Chromium and fonts

A separate Python 3.12 environment (`/content/venv`) with the exact package versions of
`requirements-pipeline.txt`, so Colab's own packages are not touched. Takes a few minutes.

In [ ]:
PY = "/content/venv/bin/python"
sh("pip install -q uv")
sh("uv venv --allow-existing --python 3.12 /content/venv")
sh(f"uv pip install --python {PY} -r {REPO}/requirements-pipeline.txt")
sh(f"{PY} -m playwright install --with-deps chromium > /tmp/playwright.log 2>&1 || (tail -30 /tmp/playwright.log; false)")

# where the generator finds SynthDoc, its Arabic add-on, OCRSmith and scan-simulator
site = subprocess.run([PY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                      capture_output=True, text=True, check=True).stdout.strip()
Path(site, "ocr_sources.pth").write_text("\n".join(str(p) for p in [
    WORK / "synthdoc" / "src", REPO / "synthdoc_arabic_ext", WORK / "ocrsmith" / "src", WORK / "scan-simulator"]) + "\n")

if MS_FONTS:
    sh("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections")
    sh("apt-get -qq install -y ttf-mscorefonts-installer > /dev/null && fc-cache -f")

# the same imports the page-making workers do
sh(f"cd {REPO} && {PY} -c \"from pipeline.doc_pipeline import DocPipeline; import synthdoc_arabic, ocrsmith, scan_simulator; print('the generator and all its parts import')\"")

## 3. The photo and text pools

Taken from `MyDrive/surya/cache/` when they are there. The first time they are built here (photos from Lorem
Picsum and Wikimedia Commons, text from Wikipedia through Hugging Face) and kept there; that takes a while
(roughly half an hour) and only happens once.

In [ ]:
CACHE = DRIVE / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
for pool, build in (("text_pool", "pipeline.text_pool --articles 5000"), ("img_pool", "pipeline.img_pool")):
    saved = CACHE / f"{pool}.tar"
    if (REPO / pool).exists():
        pass
    elif saved.exists():
        sh(f"tar -xf {saved} -C {REPO}")
    else:
        sh(f"cd {REPO} && {PY} -m {build}")
        sh(f"tar -cf {saved} -C {REPO} {pool}")
    print(pool, sum(1 for f in (REPO / pool).rglob("*") if f.is_file()), "files")

## 4. A test: four pages

Makes 4 pages and shows one, to see that everything works before the long run.

In [ ]:
import json, os, time
from IPython.display import Image as Show, display

TEST = Path("/content/test")
sh(f"rm -rf {TEST}")
start = time.time()
sh(f"cd {REPO} && {PY} -m pipeline.dataset --n 4 --seed 999 --out {TEST} --task {TASK} --tashkeel {TASHKEEL} --workers 2")
rows = [json.loads(line) for line in (TEST / "manifest.jsonl").read_text(encoding="utf-8").splitlines()]
if not rows:
    raise RuntimeError("No test page was made; why is in the output above.")
print(f"{len(rows)} of 4 pages in {time.time() - start:.0f} s")
row = rows[0]
print(row["task"], row["lang"], "| prompt:", (TEST / row["prompts"]).read_text(encoding="utf-8"))
print("target:", (TEST / row["targets"]).read_text(encoding="utf-8")[:600])
display(Show(filename=str(TEST / row["images"]), width=500))

## 5. Make the batches

One folder per epoch, made in `/content/batches/` and then copied to `MyDrive/surya/batches/`. `dataset.json`
is copied last: the fine-tuning notebook takes a folder once that file is there. A folder already in Drive is
skipped, so after a disconnect *Run all* continues with the next one (a folder that was half made starts again).

In [ ]:
import shutil

workers = WORKERS or os.cpu_count()
for epoch in EPOCHS:
    name = f"epoch_{epoch:03d}"
    local, saved = Path("/content/batches") / name, DRIVE / "batches" / name
    if (saved / "dataset.json").exists():
        print(name, "is already in Drive")
        continue
    start = time.time()
    sh(f"cd {REPO} && {PY} -m pipeline.dataset --n {PAGES} --seed {epoch} --out {local} --task {TASK} --tashkeel {TASHKEEL} --workers {workers}")
    made = len((local / "manifest.jsonl").read_text(encoding="utf-8").splitlines())
    if made < 0.95 * PAGES:
        raise RuntimeError(f"{name}: only {made} of {PAGES} pages were made; why is in the output above")
    shutil.copytree(local, saved, ignore=shutil.ignore_patterns("dataset.json"), dirs_exist_ok=True)
    shutil.copy2(local / "dataset.json", saved / "dataset.json")
    pages = len((saved / "manifest.jsonl").read_text(encoding="utf-8").splitlines())
    print(f"{name}: {pages} pages in {(time.time() - start) / 60:.0f} min -> {saved}")